# Main final report

## Project objective

Predict a continuous SHL grammar score from spoken responses while preserving the current project methodology and filename alignment. This notebook is the final training-side report: it documents the selected representation and models, reports the saved cross-validation result, and computes the compulsory in-sample training RMSE from the already-fitted artifact.

No model is retrained here. The reusable feature construction and training logic remain in `scripts/`; this notebook loads their saved outputs and artifact.

# 13. Final model
Fit the selected validated pipeline on all available training rows and report training RMSE. The selected artifact is the validated 50/50 Ridge plus HistGradientBoosting ensemble.

In [ ]:
from pathlib import Path
import sys
import joblib
import numpy as np
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from scripts.pipeline_common import feature_matrix, load_feature_sets
frames = load_feature_sets()
artifact = joblib.load(ROOT / 'outputs' / 'final_model.joblib')
x, y = feature_matrix(artifact['ridge_features'], frames)
if artifact['kind'] == 'ensemble':
    prediction = artifact['ridge_weight'] * artifact['ridge'].predict(x) + (1 - artifact['ridge_weight']) * artifact['nonlinear'].predict(x)
else:
    prediction = artifact['pipeline'].predict(x[artifact['columns']])
training_rmse = float(np.sqrt(np.mean((y.to_numpy() - prediction) ** 2)))
print('rows:', len(y), 'training RMSE:', training_rmse)
print('artifact:', ROOT / 'outputs' / 'final_model.joblib')

## Final approach and saved validation result

The final feature set is `original_linguistic+acoustic`. The pipeline imputes missing numeric values inside each estimator; the Ridge component standardizes its inputs and uses `alpha=30`; the nonlinear component is the saved `HistGradientBoostingRegressor` pipeline. The ensemble averages the two predictions with Ridge weight `0.5` and HistGradientBoosting weight `0.5`.

The saved 5-fold cross-validation result is RMSE `0.7731985964727636` and Pearson `0.7812503393565036`. The selected components scored RMSE `0.8151173374725303` / Pearson `0.7527653760330594` for Ridge and RMSE `0.7898459026180773` / Pearson `0.7715411286723808` for HistGradientBoosting. These are validation metrics, distinct from the training RMSE computed below.

## Model comparison

The experiment table in `outputs/experiment_results.csv` is the source of truth for all tested feature sets and models. It includes Ridge alpha sweeps, acoustic and linguistic alternatives, V2 alternatives, nonlinear models, and the selected ensemble.

In [ ]:
# Load saved experiment results and visualize the already-computed training predictions.
# This cell uses the fitted artifact; it does not retrain either component.
import pandas as pd
import matplotlib.pyplot as plt

results = pd.read_csv(ROOT / "outputs" / "experiment_results.csv")
print(results.sort_values(["rmse", "pearson"], ascending=[True, False]).to_string(index=False))

print("Training RMSE (in-sample):", training_rmse)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].scatter(y, prediction, alpha=0.55)
axes[0].plot([y.min(), y.max()], [y.min(), y.max()], linestyle="--", color="black")
axes[0].set_xlabel("Actual grammar score")
axes[0].set_ylabel("Predicted grammar score")
axes[0].set_title("Predicted vs actual")

residuals = y.to_numpy() - prediction
axes[1].hist(residuals, bins=20)
axes[1].set_xlabel("Actual - predicted")
axes[1].set_ylabel("Count")
axes[1].set_title("Training residuals")

axes[2].hist(y, bins=20, alpha=0.7, label="Actual")
axes[2].hist(prediction, bins=20, alpha=0.7, label="Predicted")
axes[2].set_xlabel("Grammar score")
axes[2].set_ylabel("Count")
axes[2].set_title("Target and prediction distributions")
axes[2].legend()

plt.tight_layout()
plt.show()

## How to interpret the training RMSE

The training RMSE printed by the earlier cell and visualized above is in-sample: the saved artifact has been fitted using all available training rows. It should not be confused with the 5-fold CV RMSE, which estimates performance on held-out folds and is the primary model-selection result.

Limitations include the modest labeled sample size, dependence on automatic Whisper transcripts, possible information loss in handcrafted summaries, and the fact that cross-validation is an estimate rather than an external test score. The final conclusion is therefore limited to this protocol: the saved linguistic-plus-acoustic ensemble is the strongest validated choice in the current experiment table and is ready to generate continuous test predictions.